In [ ]:
!pip uninstall -y pydrive2 cryptography pyOpenSSL
!pip install snowflake-connector-python
!pip install streamlit


Found existing installation: cryptography 46.0.3
Uninstalling cryptography-46.0.3:
  Successfully uninstalled cryptography-46.0.3
Found existing installation: pyOpenSSL 25.3.0
Uninstalling pyOpenSSL-25.3.0:
  Successfully uninstalled pyOpenSSL-25.3.0
  Using cached cryptography-46.0.3-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (5.7 kB)
  Using cached pyopenssl-25.3.0-py3-none-any.whl.metadata (17 kB)
Using cached cryptography-46.0.3-cp311-abi3-manylinux_2_34_x86_64.whl (4.5 MB)
Using cached pyopenssl-25.3.0-py3-none-any.whl (57 kB)


In [ ]:
import snowflake.connector
import pandas as pd
from getpass import getpass

# 1. Setup Connection
conn = snowflake.connector.connect(
    user='',
    password=getpass("Enter Snowflake Password: "),
    account='',
    warehouse='COMPUTE_WH',
    database='STATSBOMB_DB',
    schema='PROD_WH_MARTS'
)

Enter Snowflake Password: ··········


In [ ]:
import snowflake.connector
import pandas as pd
from getpass import getpass

# 1. Setup Connection
conn = snowflake.connector.connect(
    user='',
    password=getpass("Enter Snowflake Password: "),
    account='',
    warehouse='COMPUTE_WH',
    database='STATSBOMB_DB',
    schema='PROD_WH_MARTS'
)

# 2. Define the Feature Engineering Query
query = """
WITH team_season_stats AS (
    SELECT
        team_id,
        team_name,
        CASE
            WHEN MONTH(match_date) >= 8
            THEN YEAR(match_date) || '/' || (YEAR(match_date) + 1)
            ELSE (YEAR(match_date) - 1) || '/' || YEAR(match_date)
        END AS season,

        COUNT(DISTINCT match_id) AS matches_played,
        AVG(possession_pct) AS avg_possession,
        AVG(total_passes) AS avg_passes,
        AVG(total_shots) AS avg_shots,
        AVG(total_team_xg) AS avg_xg,
        AVG(total_pressures) AS avg_pressures,

        AVG(total_team_xg / NULLIF(total_shots, 0)) AS avg_shot_quality,
        AVG(total_passes / NULLIF(possession_pct, 0)) AS passing_intensity,
        AVG(total_pressures / NULLIF(possession_pct, 0)) AS pressing_intensity

    FROM PROD_WH_MARTS.fct_team_match_stats
    GROUP BY 1, 2, 3
    HAVING matches_played >= 5
),
network_features AS (
    SELECT
        pn.team_id,
        CASE
            WHEN MONTH(m.match_date) >= 8
            THEN YEAR(m.match_date) || '/' || (YEAR(m.match_date) + 1)
            ELSE (YEAR(m.match_date) - 1) || '/' || YEAR(m.match_date)
        END AS season,

        COUNT(DISTINCT passer_player_id) AS unique_passers,
        AVG(pass_count) AS avg_connection_strength,
        STDDEV(passer_x) AS team_width,
        STDDEV(passer_y) AS team_depth

    FROM PROD_WH_MARTS.fct_passing_network pn
    LEFT JOIN STATSBOMB_DB.PROD_WH.DIM_MATCH m ON pn.match_id = m.match_id
    GROUP BY 1, 2
)

SELECT
    tss.*,
    COALESCE(nf.unique_passers, 0) as unique_passers,
    COALESCE(nf.avg_connection_strength, 0) as avg_connection_strength,
    COALESCE(nf.team_width, 0) as team_width,
    COALESCE(nf.team_depth, 0) as team_depth
FROM team_season_stats tss
LEFT JOIN network_features nf
    ON tss.team_id = nf.team_id
    AND tss.season = nf.season
ORDER BY tss.season DESC, tss.team_name;
"""

# 3. Load into Pandas
print("Fetching data from Snowflake...")
df = pd.read_sql(query, conn)
print(f"Data Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Enter Snowflake Password: ··········
Fetching data from Snowflake...


/tmp/ipython-input-1251239328.py:76: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Data Loaded: 237 rows, 16 columns


,TEAM_ID,TEAM_NAME,SEASON,MATCHES_PLAYED,AVG_POSSESSION,AVG_PASSES,AVG_SHOTS,AVG_XG,AVG_PRESSURES,AVG_SHOT_QUALITY,PASSING_INTENSITY,PRESSING_INTENSITY,UNIQUE_PASSERS,AVG_CONNECTION_STRENGTH,TEAM_WIDTH,TEAM_DEPTH
0,865,England Women's,2024/2025,6,54.705000,1934.333333,19.166667,4.014439,64.666667,0.205363,36.242901,1.271885,18,6.437870,17.930697,17.876310
1,857,Germany Women's,2024/2025,5,52.118000,1640.800000,18.000000,2.683320,68.600000,0.164786,34.075326,1.631164,19,6.167442,20.168001,17.302124
2,855,Italy Women's,2024/2025,5,42.752000,1340.400000,13.800000,1.258393,46.800000,0.092396,32.268226,1.101647,16,4.835749,17.362944,18.604853
3,863,Spain Women's,2024/2025,6,70.536667,2459.500000,24.833333,3.173782,73.000000,0.127771,35.162184,1.048828,19,8.274194,16.689948,17.648405
4,4901,Angola,2023/2024,5,45.432000,1362.600000,10.800000,1.333232,49.200000,0.132055,30.540774,1.099404,20,4.984925,16.563196,19.184796


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Select features for clustering
features = [
    'AVG_POSSESSION', 'AVG_PASSES', 'AVG_SHOTS', 'AVG_XG',
    'PRESSING_INTENSITY', 'AVG_SHOT_QUALITY', 'TEAM_WIDTH'
]

# Handle any remaining nulls
X = df[features].fillna(0)

# 2. Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Train K-Means (K=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X_scaled)

# 4. Assign Clusters back to the main DataFrame
df['CLUSTER'] = kmeans.labels_

print("Clustering Complete.")
print(df['CLUSTER'].value_counts())

Clustering Complete.
CLUSTER
1    105
2     66
3     44
0     22
Name: count, dtype: int64


In [ ]:
# 1. Calculate the mean of features for each cluster to interpret them
cluster_summary = df.groupby('CLUSTER')[features].mean().reset_index()

print("Cluster Characteristics:")
display(cluster_summary)

# 2. Define labels based on the summary (Update this based on what you see above!)
# Example Logic:
# If Cluster 0 has high Possession/Passes -> "Possession Control"
# If Cluster 1 has high Pressing -> "High Press"
# If Cluster 2 has low Passes but High Shot Quality -> "Counter Attack"

# Map the labels
style_map = {
    0: 'Balanced',
    1: 'Counter-Attacking/ Direct Playing',
    2: 'Slow-Tempo Positional Teams',
    3: '"High-Possession Dominators'
}

df['STYLE_NAME'] = df['CLUSTER'].map(style_map)


# View a few examples
df[['TEAM_NAME', 'SEASON', 'STYLE_NAME', 'AVG_POSSESSION', 'PRESSING_INTENSITY']].head(20)



Cluster Characteristics:


,CLUSTER,AVG_POSSESSION,AVG_PASSES,AVG_SHOTS,AVG_XG,PRESSING_INTENSITY,AVG_SHOT_QUALITY,TEAM_WIDTH
0,0,51.460655,1711.964719,14.480628,2.473588,1.103291,0.163572,17.132955
1,1,45.540463,1463.288308,11.162272,1.094149,1.288274,0.097515,16.513533
2,2,55.378129,1723.630704,14.389117,1.529155,1.048781,0.106339,17.228854
3,3,63.391888,2148.471456,17.657155,2.262529,0.927231,0.127897,17.171452


,TEAM_NAME,SEASON,STYLE_NAME,AVG_POSSESSION,PRESSING_INTENSITY
0,England Women's,2024/2025,Balanced,54.705000,1.271885
1,Germany Women's,2024/2025,Balanced,52.118000,1.631164
2,Italy Women's,2024/2025,Counter-Attacking/ Direct Playing,42.752000,1.101647
3,Spain Women's,2024/2025,"""High-Possession Dominators",70.536667,1.048828
4,Angola,2023/2024,Counter-Attacking/ Direct Playing,45.432000,1.099404
5,Argentina,2023/2024,Balanced,57.606667,0.940679
6,Bayer Leverkusen,2023/2024,"""High-Possession Dominators",61.204412,0.896226
7,Canada,2023/2024,Balanced,47.256667,1.218648
8,Cape Verde Islands,2023/2024,Slow-Tempo Positional Teams,52.558000,0.901202
9,Colombia,2023/2024,Slow-Tempo Positional Teams,53.318333,1.036177


In [ ]:
filtered_df = df[df['TEAM_NAME'].isin(['Barcelona', 'Atlético Madrid', 'Manchester City', 'FC Barcelona', 'Atlético de Madrid', 'Manchester City'])]
display(filtered_df)

,TEAM_ID,TEAM_NAME,SEASON,MATCHES_PLAYED,AVG_POSSESSION,AVG_PASSES,AVG_SHOTS,AVG_XG,AVG_PRESSURES,AVG_SHOT_QUALITY,PASSING_INTENSITY,PRESSING_INTENSITY,UNIQUE_PASSERS,AVG_CONNECTION_STRENGTH,TEAM_WIDTH,TEAM_DEPTH,CLUSTER,STYLE_NAME
55,217,Barcelona,2020/2021,35,63.228857,2376.914286,15.514286,2.057723,55.028571,0.137241,37.773448,0.885574,25,7.341808,17.786890,16.356329,3,"""High-Possession Dominators"
75,217,Barcelona,2019/2020,33,65.894545,2359.272727,13.030303,1.676482,56.636364,0.131401,36.000168,0.869534,27,7.449391,17.838074,16.432358,3,"""High-Possession Dominators"
88,217,Barcelona,2018/2019,34,63.792353,2275.705882,15.264706,1.829217,58.941176,0.116764,35.838221,0.945475,24,7.270552,18.307032,16.694499,3,"""High-Possession Dominators"
107,217,Barcelona,2017/2018,36,61.333611,2140.305556,15.194444,2.077212,60.972222,0.134960,35.644109,1.014970,24,7.164848,18.263961,17.674877,3,"""High-Possession Dominators"
124,217,Barcelona,2016/2017,34,66.318529,2150.500000,16.588235,2.179229,58.382353,0.133021,32.554641,0.903712,24,7.361738,19.298287,17.600631,3,"""High-Possession Dominators"
134,212,Atlético Madrid,2015/2016,39,47.692821,1690.051282,12.948718,1.503719,69.384615,0.115236,35.988170,1.498576,25,5.788480,14.947376,17.558737,1,Counter-Attacking/ Direct Playing
136,217,Barcelona,2015/2016,38,66.886579,2169.447368,15.894737,2.406148,55.578947,0.153340,32.579169,0.845444,24,7.326652,17.977211,17.389218,3,"""High-Possession Dominators"
181,36,Manchester City,2015/2016,38,56.696316,1917.131579,16.157895,1.698817,59.894737,0.104165,33.925453,1.076993,24,6.288329,16.190585,18.074171,2,Slow-Tempo Positional Teams
223,217,Barcelona,2014/2015,39,69.170256,2319.948718,16.461538,2.214097,60.589744,0.133955,33.568505,0.888039,25,7.373819,17.049885,16.824148,3,"""High-Possession Dominators"
224,217,Barcelona,2013/2014,31,67.717419,2294.806452,17.322581,2.066301,61.290323,0.119155,34.089473,0.916013,23,7.501102,17.112387,16.285436,3,"""High-Possession Dominators"


In [ ]:
from snowflake.connector.pandas_tools import write_pandas

# 1. Prepare the dataframe (Ensure column names are upper case for Snowflake)
df.columns = [c.upper() for c in df.columns]

# 2. Write to Snowflake
# This creates the table 'TEAM_STYLES_FINAL' that your App (Part 2) needs
success, n_chunks, n_rows, _ = write_pandas(
    conn,
    df,
    "TEAM_STYLES_FINAL",
    database="STATSBOMB_DB",
    schema="PROD_WH_MARTS",
    auto_create_table=True,
    overwrite=True  # Overwrites the table every time you re-run the model
)

print(f"Success! {n_rows} rows written to Snowflake. Now run the App.")

Success! 286 rows written to Snowflake. Now run the App.


In [ ]:
import streamlit as st
import pandas as pd

# 1. Page Configuration
st.set_page_config(
    page_title="Football Tactical Engine",
    page_icon="⚽",
    layout="wide"
)

st.title("⚽ Tactical Scouting Engine")
st.markdown("Find managers who play exactly like your target.")



# 3. Load Data (Cached!)
# We use @st.cache_data so we don't bill Snowflake for every single click.
# The 'ttl=3600' means it re-runs the query once every hour.
@st.cache_data(ttl=3600)
def load_data():
    query = """
    SELECT
        TEAM_NAME,
        SEASON,
        -- Create a display name like "Guardiola (Man City 2023)"
        TEAM_NAME || ' (' || SEASON || ')' AS MANAGER_ID,
        AVG_POSSESSION,
        PRESSING_INTENSITY,
        AVG_PASSES,
        AVG_XG,
        -- Assuming you have the CLUSTER column from your previous python/sql step
        CLUSTER
    FROM PROD_WH_MARTS.TEAM_STYLES_FINAL
    WHERE MATCHES_PLAYED > 10
    """
    return conn.query(query)

# Load the data
try:
    df = load_data()

    # 4. The User Interface

    # Sidebar for Inputs
    with st.sidebar:
        st.header("Search Parameters")

        # Dropdown to select a team/manager
        # We use the unique MANAGER_ID we created in the SQL
        all_teams = df['MANAGER_ID'].unique()
        selected_team = st.selectbox("Select Target Team/Manager:", all_teams)

        find_btn = st.button("Find Similar Teams", type="primary")

    # 5. The Logic
    if find_btn:
        # A. Find the cluster of the selected team
        target_row = df[df['MANAGER_ID'] == selected_team].iloc[0]
        target_cluster = target_row['CLUSTER']
        target_possession = target_row['AVG_POSSESSION']

        # B. Filter for same cluster, excluding the selected team itself
        similar_teams = df[
            (df['CLUSTER'] == target_cluster) &
            (df['MANAGER_ID'] != selected_team)
        ].copy()

        # C. Calculate specific similarity (Distance to target possession)
        # This sorts them by how close their possession stats are to the target
        similar_teams['Similarity_Diff'] = abs(similar_teams['AVG_POSSESSION'] - target_possession)
        similar_teams = similar_teams.sort_values('Similarity_Diff').head(10)

        # 6. Display Results
        st.subheader(f"Teams playing like {selected_team}")
        st.caption(f"Tactical Cluster: {target_cluster} | Target Possession: {target_possession:.1f}%")

        # Show Metrics
        col1, col2, col3 = st.columns(3)
        col1.metric("Matches Found", len(similar_teams))
        col2.metric("Avg Possession (Group)", f"{similar_teams['AVG_POSSESSION'].mean():.1f}%")
        col3.metric("Avg Pressing (Group)", f"{similar_teams['PRESSING_INTENSITY'].mean():.2f}")

        # Display the Table
        st.dataframe(
            similar_teams[['MANAGER_ID', 'AVG_POSSESSION', 'PRESSING_INTENSITY', 'AVG_XG']],
            use_container_width=True,
            hide_index=True
        )

        # Display Chart
        st.subheader("Possession Comparison")
        st.bar_chart(similar_teams.set_index('MANAGER_ID')['AVG_POSSESSION'])

except Exception as e:
    st.error(f"Error connecting to Snowflake. Check your secrets.toml file.\n\nDetails: {e}")



ModuleNotFoundError: No module named 'streamlit'